<a href="https://colab.research.google.com/github/rsher60/LLM_Codebase/blob/main/fine_tuning_Ed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# imports

import os
import random
from dotenv import load_dotenv
from huggingface_hub import login
from datasets import load_dataset, Dataset, DatasetDict

from items import Item

import matplotlib.pyplot as plt
from collections import Counter, defaultdict
import numpy as np
import pickle

In [2]:
!pip install -q typing transformers datasets python_dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 16.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [6]:
from typing import Optional
from transformers import AutoTokenizer
import re
from datasets import load_dataset

In [62]:
data = load_dataset("pgurazada1/amazon_india_products")["train"]

In [63]:
# Investigate a particular datapoint
datapoint = data[2]

In [64]:
datapoint

{'Uniq Id': '41654633cce38c8650690f6dbac01fd3',
 'Crawl Timestamp': '2019-10-30 09:53:23 +0000',
 'Category': 'Skin Care',
 'Product Title': ' Generic 1 Pc brand snail eye cream remove dark circle eye lifting instant ageless snail cream for eye care anti wrinkle eyes cream 20g ',
 'Product Description': 'Use: eye, item type: cream, net wt: 20g, gzzz: ygzwbz, model number: lkwnys, gender: female, certification: gzzz, feature: moisturizing, dark circle, anti-puffiness, anti-aging, brand name: laikou, certificate number: 2014028630, ingredient: cream, country/region of manufacture: china, eye care features: eye cream, product name: snail extract cream, applicable to the crowd: general, specifications: normal specifications, origin: shantou, guangdong province, unit type: piece, package weight: ,package size:',
 'Brand': 'Generic',
 'Pack Size Or Quantity': None,
 'Mrp': '1824.00',
 'Price': '1042.00',
 'Site Name': 'Amazon In',
 'Offers': '42.87%',
 'Combo Offers': None,
 'Stock Availibil

In [65]:
# How many have prices?

prices = 0
for datapoint in data:
    try:
        price = float(datapoint["Mrp"] or 0.0)
        if price > 0.0:
            prices += 1
    except ValueError as e:
        pass

print(f"There are {prices:,} with prices which is {prices/len(data)*100:,.1f}%")

There are 29,240 with prices which is 97.5%


In [78]:
import pandas as pd
df  = pd.DataFrame(data)

In [79]:
#Drop the rows where the price is none and then delete rows where price is 0.0

df = df.dropna(subset=['Mrp'])



In [80]:
df.dtypes

,0
Uniq Id,object
Crawl Timestamp,object
Category,object
Product Title,object
Product Description,object
Brand,object
Pack Size Or Quantity,object
Mrp,object
Price,object
Site Name,object


In [84]:
clean_price('2.4.00')

'24.00'

In [85]:
# CHange the data tyoe of the column to float

import re

def clean_price(price_str):
  """Cleans a price string by removing extra decimal points."""
  if price_str.count('.') ==2:
    cleaned_price = re.sub(r"\.", "", price_str.strip(), count=1)
    return cleaned_price
  else:
    return price_str


In [86]:
# apply the clean price function

df['Mrp'] = df['Mrp'].apply(lambda x : float(clean_price(x) or 0.0))


,0
Uniq Id,object
Crawl Timestamp,object
Category,object
Product Title,object
Product Description,object
Brand,object
Pack Size Or Quantity,object
Mrp,float64
Price,object
Site Name,object


In [61]:
#count the number of .

dr = 'dfbfddf.232.'


dr.count('.')


2

In [60]:
df.head(40)

,Uniq Id,Crawl Timestamp,Category,Product Title,Product Description,Brand,Pack Size Or Quantity,Mrp,Price,Site Name,Offers,Combo Offers,Stock Availibility,Product Asin,Image Urls
0,eb49cc038190f6f03c272f79fbbce894,2019-10-30 11:38:11 +0000,Skin Care,Lee posh Lactic Acid 60% Anti ageing Pigmenta...,PROFESSIONAL GRADE Face Peel: this peel stimul...,Lee Posh,None,200000.0,799.00,Amazon In,60.05%,None,YES,B072BGHNJ1,https://images-na.ssl-images-amazon.com/images...
1,1657cc30c438affede6a5060d6847363,2019-10-31 15:46:54 +0000,Skin Care,Branded SLB Works New 1.5mm Titanium 1200 nee...,Item name: 1.5mm titanium 1200 needles microne...,SLB Works,None,204000.0,2040.00,Amazon In,0%,None,YES,B07QDTZYSJ,https://images-na.ssl-images-amazon.com/images...
2,41654633cce38c8650690f6dbac01fd3,2019-10-30 09:53:23 +0000,Skin Care,Generic 1 Pc brand snail eye cream remove dar...,"Use: eye, item type: cream, net wt: 20g, gzzz:...",Generic,None,182400.0,1042.00,Amazon In,42.87%,None,YES,B07DCSN8MP,https://images-na.ssl-images-amazon.com/images...
3,08b1bd85c3efc2d7aa556fd79b073382,2019-10-29 16:16:52 +0000,Skin Care,Generic Anti Snoring Snore Stopper Sleep Apne...,Prevent the tongue from dropping backward or b...,Generic,None,218500.0,1399.00,Amazon In,35.97%,None,YES,B07GLW9VQN,https://images-na.ssl-images-amazon.com/images...
4,3ac3f213732512d1d11bb73ab3b1900f,2019-10-31 09:32:06 +0000,Grocery & Gourmet Foods,Harveys Crunchy & Creame Gourmet Delicacies C...,Harvey's wafer Cream Wafer 110g. Made in India,Harveys,None,59400.0,570.00,Amazon In,4.04%,None,YES,B07NFYYLF1,https://images-na.ssl-images-amazon.com/images...
5,f89b246d4e27c11623dbc7742523f319,2019-10-30 19:14:20 +0000,Skin Care,"Shikai Borage Dry Skin Therapy Foot Cream, 4....","Package Quantity:3 Contains borage oil, clinic...",ShiKai,354 g,534400.0,5344.00,Amazon In,0%,None,YES,B001ET7E8C,https://images-na.ssl-images-amazon.com/images...
6,ddb01f73b53ea264c0a9fcc31b1dbd72,2019-10-30 07:46:44 +0000,Bath & Shower,Black & Tan Beer Soap 4-Pack,Our handmade soaps are made with the highest g...,Lather+%26+Fizz+Bath+Boutique,None,908600.0,7269.00,Amazon In,20.0%,None,YES,B06XSRZV65,https://images-na.ssl-images-amazon.com/images...
7,516c5986d5dc20efaa37a58cbdea755a,2019-10-31 07:16:31 +0000,Bath & Shower,Mydio 2 Pack Waterproof Women Shower Caps Bat...,Mydio 2 Pack Waterproof Women Shower Caps Bath...,Mydio,49.9 g,165900.0,1659.00,Amazon In,0%,None,YES,B074FVJN7D,https://images-na.ssl-images-amazon.com/images...
8,fc4d5d08dcad5e10f3408e89990065a0,2019-10-28 16:33:56 +0000,Grocery & Gourmet Foods,Food Studio Premium Quality in Shell Pistachi...,Pistachios are small in appearance and are enc...,FOODSTUDIO,399 Grams,76000.0,660.00,Amazon In,13.16%,None,YES,B07Y3PTX42,https://images-na.ssl-images-amazon.com/images...
9,306bc2d444aa62cd53eb46fdbfd5c6f5,2019-10-28 16:57:50 +0000,Grocery & Gourmet Foods,Dr Glutens Gluten Free Khracker (KHAKHRA) - 2...,GLUTEN FREE KHAKHRAS. LIGHT HEALTHY AND ROASTE...,Dr Gluten,None,27900.0,279.00,Amazon In,0%,None,YES,B07Y4SSRR1,https://images-na.ssl-images-amazon.com/images...


In [30]:
df['Mrp'] = df['Mrp'].astype('float64')

ValueError: could not convert string to float: '.250.00'

In [14]:
df.shape

(29400, 15)

In [45]:
# change the datatyoe of the Price column to be numberic

,0
Uniq Id,object
Crawl Timestamp,object
Category,object
Product Title,object
Product Description,object
Brand,object
Pack Size Or Quantity,object
Mrp,object
Price,object
Site Name,object


In [44]:
import pandas as pd
import matplotlib.pyplot as plt

# Create a series
series = df['Price']

# Plot a histogram
series.plot.hist()
plt.show()

TypeError: no numeric data to plot

In [6]:
df.columns

Index(['Uniq Id', 'Crawl Timestamp', 'Category', 'Product Title',
       'Product Description', 'Brand', 'Pack Size Or Quantity', 'Mrp', 'Price',
       'Site Name', 'Offers', 'Combo Offers', 'Stock Availibility',
       'Product Asin', 'Image Urls'],
      dtype='object')

In [12]:
items = ItemLoader("Amazon_India_Products").load()

NameError: name 'ItemLoader' is not defined

In [ ]:
BASE_MODEL = "meta-llama/Meta-Llama-3.1-8B"

In [ ]:
MIN_TOKENS = 150
MAX_TOKENS = 160

MIN_CHARS = 300
CEILING_CHARS = MAX_TOKENS * 7